# Azure Log Sanitization Notebook (No IP Processing)
This notebook sanitizes **Azure-exported CSV logs** across 7 tables:
- `conn`, `events`, `perf`, `port`, `process`, `security`, `syslog`

Goals:
1. Basic data-quality cleanup (deduplication, timestamp parsing, consistent columns)
2. **Stable pseudonymization** for *non-IP* identifiers (e.g., usernames, hostnames, FQDNs, Azure Resource IDs, GUIDs)
3. Output sanitized datasets to `sanidata/`

### Important constraint
**We do NOT modify, hash, mask, or otherwise process any IP addresses** in this version.
IPs are kept exactly as-is in all columns and in free-text fields.

Recommendation:
- Keep raw inputs in `data/` (private)
- Keep mapping files in `mappings/` and add that folder to `.gitignore`.


In [10]:
# --- 0) Imports & Paths ---
import os, re, json, hashlib
import pandas as pd
from pathlib import Path
from collections import defaultdict

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 140)

BASE_DIR = Path.cwd().parent.resolve().joinpath("data")

# Input files
FILES = {
    "conn":    "azure_conn.csv",
    "events":  "azure_events.csv",
    "perf":    "azure_perf.csv",
    "port":    "azure_port.csv",
    "process": "azure_process.csv",
    "security":"azure_security.csv",
    "syslog":  "syslog.csv",
}

def extract_scenario(path: Path) -> str:
    """
    Extract scenario id like 'sc1' from any part of the path.
    Falls back to 'unknown' if not found.
    """
    for part in path.parts:
        if part.startswith("sc") and part[2:].isdigit():
            return part
    return "unknown"

def collect_inputs(base: Path, files: dict) -> dict:
    """
    Return:
      inputs[scenario][table] -> list[Path]  (can be 1 or many)
    We keep lists to support multiple occurrences per scenario if they exist.
    """
    inputs = defaultdict(lambda: defaultdict(list))
    for table, fname in files.items():
        for p in base.rglob(fname):
            sc = extract_scenario(p)
            inputs[sc][table].append(p)
    return inputs

def collect_inputs(base: Path, files: dict) -> dict:
    """
    Return:
      inputs[scenario][table] -> list[Path]  (can be 1 or many)
    We keep lists to support multiple occurrences per scenario if they exist.
    """
    inputs = defaultdict(lambda: defaultdict(list))
    for table, fname in files.items():
        for p in base.rglob(fname):
            sc = extract_scenario(p)
            inputs[sc][table].append(p)
    return inputs

INPUTS = collect_inputs(BASE_DIR, FILES)

# Quick sanity print: how many matches per scenario/table
for sc in sorted(INPUTS):
    counts = {t: len(ps) for t, ps in INPUTS[sc].items()}
    print(f"{sc}: {counts}")

OUT_DIR = BASE_DIR.parent.resolve().joinpath("sanidata")
MAP_DIR = BASE_DIR.parent.resolve().joinpath("mappings") 
OUT_DIR.mkdir(parents=True, exist_ok=True)
MAP_DIR.mkdir(parents=True, exist_ok=True)

# A stable SALT is used to derive deterministic tokens across runs.
# Keep this secret. Do NOT publish it.
SALT_PATH = MAP_DIR / "salt.txt"
if SALT_PATH.exists():
    SALT = SALT_PATH.read_text().strip()
else:
    SALT = hashlib.sha256(os.urandom(32)).hexdigest()
    SALT_PATH.write_text(SALT)

print("Base data dir:", BASE_DIR.resolve())
print("Output base dir:", OUT_DIR.resolve())
print("Mapping dir:", MAP_DIR.resolve())


sc1: {'conn': 1, 'events': 1, 'perf': 1, 'port': 1, 'process': 1, 'security': 1}
sc2: {'events': 1, 'syslog': 1}
sc3: {'syslog': 1}
sc4: {'syslog': 1}
sc5: {'events': 1}
sc6: {'events': 1, 'syslog': 1}
sc7: {'syslog': 1}
Base data dir: /Users/zhuoran/Projects/SSCMDataset/data
Output base dir: /Users/zhuoran/Projects/SSCMDataset/sanidata
Mapping dir: /Users/zhuoran/Projects/SSCMDataset/mappings


## 1) Stable pseudonymization primitives
We map sensitive identifiers to deterministic tokens (e.g., `USER_001`, `HOST_042`) so that:
- The same real-world entity maps to the same token across all tables.
- The sanitized tables remain joinable for cross-log correlation.

Mapping tables are stored as JSON under `mappings/`.

In [11]:
# --- 1) Mapping helpers (stable across runs) ---

def _load_map(name: str) -> dict:
    p = MAP_DIR / f"{name}.json"
    if p.exists():
        return json.loads(p.read_text())
    return {}

def _save_map(name: str, m: dict):
    p = MAP_DIR / f"{name}.json"
    p.write_text(json.dumps(m, ensure_ascii=False, indent=2))

def stable_hash(text: str) -> str:
    return hashlib.sha256((SALT + str(text)).encode("utf-8")).hexdigest()

def map_token(value, prefix: str, map_name: str, digits: int = 3):
    """Map any value to a stable token PREFIX_XXX, persisted in mappings/<map_name>.json."""
    if pd.isna(value) or value == "":
        return value
    value = str(value)
    m = _load_map(map_name)
    if value in m:
        return m[value]

    # Derive a pseudo-random but deterministic numeric id from the salted hash
    base = int(stable_hash(value)[:8], 16) % (10**digits)
    candidate = f"{prefix}_{base:0{digits}d}"

    # Resolve collisions (rare but possible) deterministically
    used = set(m.values())
    i = 0
    while candidate in used:
        i += 1
        candidate = f"{prefix}_{(base+i)%(10**digits):0{digits}d}"

    m[value] = candidate
    _save_map(map_name, m)
    return candidate


## 2) Regex-based sanitizers (non-IP)
We sanitize the following patterns:
- **Azure Resource IDs**: `/subscriptions/<...>/resourceGroups/<...>/...`
- **GUIDs**: `xxxxxxxx-xxxx-xxxx-xxxx-xxxxxxxxxxxx`
- **FQDNs / hostnames in DNS fields**

Again: **IPs are intentionally not sanitized**.

In [15]:
# --- 2) Non-IP regex-based sanitizers ---

GUID_RE = re.compile(r"\b[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}\b")
FQDN_RE = re.compile(r"\b(?:[a-zA-Z0-9-]+\.)+[a-zA-Z]{2,}\b")
EXCLUDE_TLDS = {"exe", "dll", "sys"}
AZ_RES_RE = re.compile(r"/subscriptions/[^/]+/resourcegroups/[^/]+/providers/[^\s\"]+", re.IGNORECASE)

# --- Path username sanitization (Windows + *nix) ---

WIN_USER_RE = re.compile(r'(?i)([A-Z]:\\Users\\)([^\\]+)(\\)')   # C:\Users\<name>\
NIX_USER_RE = re.compile(r'(/home/)([^/]+)(/)')                 # /home/<name>/

def _is_empty(x) -> bool:
    return x is None or (isinstance(x, float) and pd.isna(x)) or str(x).strip() == ""

def sanitize_guid(text: str) -> str:
    if _is_empty(text):
        return text

    if pd.isna(text): 
        return text
    s = str(text)
    return GUID_RE.sub(lambda m: map_token(m.group(0).lower(), "GUID", "guid_map", digits=4), s)

def sanitize_resource_id(text: str) -> str:
    """Sanitize Azure Resource ID while preserving structure for analysis."""
    if _is_empty(text):
        return text

    if pd.isna(text):
        return text
    s = str(text)

    def repl(m):
        rid = m.group(0)

        # Replace subscription GUIDs
        rid2 = GUID_RE.sub(lambda mm: map_token(mm.group(0).lower(), "SUB", "sub_map", digits=3), rid)

        # Replace resource group names
        rid2 = re.sub(r"(?i)/resourcegroups/([^/]+)", 
                      lambda mm: f"/resourceGroups/{map_token(mm.group(1), 'RG', 'rg_map', digits=3)}", rid2)

        # Replace VM names
        rid2 = re.sub(r"(?i)/virtualmachines/([^/]+)", 
                      lambda mm: f"/virtualMachines/{map_token(mm.group(1), 'VM', 'vm_map', digits=3)}", rid2)

        return rid2

    return AZ_RES_RE.sub(repl, s)

def sanitize_fqdn(text: str, keep_domain: bool = False) -> str:
    """Sanitize FQDNs. 
    - keep_domain=False: replace the full FQDN with a token + '.example'
    - keep_domain=True: replace only the host label, keep the domain intact
    """
    if _is_empty(text):
        return text
    if pd.isna(text):
        return text
    s = str(text)

    def repl(m):
        fqdn = m.group(0)
        parts = fqdn.split(".")
        tld = parts[-1].lower()
        if tld in EXCLUDE_TLDS:
            return fqdn
        if keep_domain and len(parts) >= 3:
            host = parts[0]
            rest = ".".join(parts[1:])
            return f"{map_token(host, 'HOST', 'host_map', digits=3)}.{rest}"
        return f"{map_token(fqdn.lower(), 'FQDN', 'fqdn_map', digits=4)}.example"

    return FQDN_RE.sub(repl, s)

def sanitize_text_blob(text: str, keep_domain: bool = False) -> str:
    """Apply sanitization to free-text fields (Message, SyslogMessage, etc.).
    Note: This function intentionally does NOT sanitize IP addresses.
    """
    if _is_empty(text):
        return text
    if pd.isna(text):
        return text
    
    s = str(text)
    s = sanitize_user_in_paths(s)
    s = sanitize_resource_id(s)
    s = sanitize_guid(s)
    s = sanitize_fqdn(s, keep_domain=keep_domain)
    return s

def sanitize_user_in_paths(text: str) -> str:
    '''
    Replace usernames embedded in file paths with stable USER tokens.
    Examples:
      C:\\Users\\Newt\\...  -> C:\\Users\\USER_001\\...
      /home/alice/...       -> /home/USER_001/...
    '''
    if _is_empty(text):
        return text
    s = str(text)

    s = WIN_USER_RE.sub(lambda m: f"{m.group(1)}{map_token(m.group(2), 'USER', 'user_map', digits=3)}{m.group(3)}", s)
    s = NIX_USER_RE.sub(lambda m: f"{m.group(1)}{map_token(m.group(2), 'USER', 'user_map', digits=3)}{m.group(3)}", s)

    return s


## 3) Load CSVs + basic data-quality cleanup
We parse timestamps (when present) and drop exact duplicates.
Timestamp formats are expected to include day-first formats such as `22/04/2025, 13:03:00.020`.


In [4]:
# --- 3) Raw load only (NO parsing / NO type conversion / NO dedup) ---

def load_csv_raw(path: Path) -> pd.DataFrame:
    """
    Load CSV without any transformation.
    Keep strings as-is to avoid affecting downstream parsing.
    """
    return pd.read_csv(path, dtype=str, keep_default_na=False)

raw = {}  # raw[scenario][table] = list[DataFrame] OR single DataFrame (your choice)

for sc, tables in INPUTS.items():
    raw[sc] = {}
    for table, paths in tables.items():
        # If you expect exactly one file per (scenario, table), keep it simple:
        if len(paths) == 1:
            raw[sc][table] = load_csv_raw(paths[0])
        else:
            # If there are multiple matches, keep them all (same logic later)
            raw[sc][table] = [load_csv_raw(p) for p in paths]

# Quick overview
summary = {
    sc: {t: (df.shape if not isinstance(df, list) else [d.shape for d in df])
         for t, df in tables.items()}
    for sc, tables in raw.items()
}
summary


{'sc1': {'conn': (5746, 44),
  'events': (1000, 21),
  'perf': (15000, 18),
  'port': (1109, 24),
  'process': (345, 27),
  'security': (334, 234)},
 'sc6': {'events': (6528, 21), 'syslog': (3246, 16)},
 'sc5': {'events': (7453, 21)},
 'sc2': {'events': (44246, 21), 'syslog': (9732, 16)},
 'sc7': {'syslog': (9805, 16)},
 'sc3': {'syslog': (14677, 16)},
 'sc4': {'syslog': (57250, 16)}}

## 4) Per-table sanitization rules (Azure)
Design principles:
- Keep semantic/security-relevant fields (event types, ports, protocols, direction, etc.)
- Pseudonymize only identifiers that may leak environment details (usernames, hostnames, FQDNs, GUIDs, Resource IDs)
- Apply regex-based sanitization on free-text columns
- **Do not touch IP addresses**


In [5]:
# --- 4) Per-table sanitizers ---

def sanitize_common_ids(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Azure / agent identifiers
    for col, pref, mname in [
        ("TenantId", "TEN", "tenant_map"),
        ("AgentId", "AGENT", "agent_map"),
        ("MG", "MG", "mg_map"),
        ("ManagementGroupName", "MGN", "mgn_map"),
    ]:
        if col in df.columns:
            df[col] = df[col].apply(lambda v: map_token(v, pref, mname, digits=3))

    # Machine / hostname-like fields
    for col in ["Computer", "HostName", "CollectorHostName", "Machine", "WorkstationName"]:
        if col in df.columns:
            df[col] = df[col].apply(lambda v: map_token(v, "HOST", "hostnames_map", digits=3))

    # Azure ResourceId
    if "_ResourceId" in df.columns:
        df["_ResourceId"] = df["_ResourceId"].apply(sanitize_resource_id)

    return df


def sanitize_conn(df: pd.DataFrame) -> pd.DataFrame:
    df = sanitize_common_ids(df)
    # DNS arrays stored as strings (e.g., ["asavpn..."])
    for col in ["RemoteDnsQuestions", "RemoteDnsCanonicalNames"]:
        if col in df.columns:
            df[col] = df[col].apply(lambda v: sanitize_text_blob(v, keep_domain=False))
    return df


def sanitize_events(df: pd.DataFrame) -> pd.DataFrame:
    df = sanitize_common_ids(df)
    for col in ["UserName", "Account", "SubjectUserName", "TargetUserName", "CallerUserName"]:
        if col in df.columns:
            df[col] = df[col].apply(lambda v: map_token(v, "USER", "user_map", digits=3))
    if "Message" in df.columns:
        df["Message"] = df["Message"].apply(lambda v: sanitize_text_blob(v, keep_domain=False))
    if "RenderedDescription" in df.columns:
        df["RenderedDescription"] = df["RenderedDescription"].apply(lambda v: sanitize_text_blob(v, keep_domain=False))
    return df


def sanitize_perf(df: pd.DataFrame) -> pd.DataFrame:
    df = sanitize_common_ids(df)
    for col in ["CounterPath"]:
        if col in df.columns:
            df[col] = df[col].apply(lambda v: sanitize_text_blob(v, keep_domain=False))
    return df


def sanitize_port(df: pd.DataFrame) -> pd.DataFrame:
    df = sanitize_common_ids(df)
    return df


def sanitize_process(df: pd.DataFrame) -> pd.DataFrame:
    df = sanitize_common_ids(df)
    for col in ["UserName", "UserDomain"]:
        if col in df.columns:
            df[col] = df[col].apply(lambda v: map_token(v, "USER", "user_map", digits=3))

    # These fields may embed paths/usernames/FQDNs/GUIDs/ResourceIds
    for col in ["CommandLine", "ExecutablePath", "ParentProcessName", "ProcessName", "Process"]:
        if col in df.columns:
            df[col] = df[col].apply(lambda v: sanitize_text_blob(v, keep_domain=False))

    return df


def sanitize_security(df: pd.DataFrame) -> pd.DataFrame:
    df = sanitize_common_ids(df)

    # Account-like identifiers
    for col in ["Account", "TargetAccount", "SubjectUserName", "TargetUserName", "UserName", "CallerUserName"]:
        if col in df.columns:
            df[col] = df[col].apply(lambda v: map_token(v, "USER", "user_map", digits=3))

    # Textual descriptions often include GUIDs/ResourceIds/FQDNs
    for col in ["RenderedDescription", "Message", "ProcessName"]:
        if col in df.columns:
            df[col] = df[col].apply(lambda v: sanitize_text_blob(v, keep_domain=False))

    return df


def sanitize_syslog(df: pd.DataFrame) -> pd.DataFrame:
    df = sanitize_common_ids(df)
    for col in ["SyslogMessage", "ProcessName"]:
        if col in df.columns:
            df[col] = df[col].apply(lambda v: sanitize_text_blob(v, keep_domain=False))
    return df


SANITIZERS = {
    "conn": sanitize_conn,
    "events": sanitize_events,
    "perf": sanitize_perf,
    "port": sanitize_port,
    "process": sanitize_process,
    "security": sanitize_security,
    "syslog": sanitize_syslog,
}


## 5) Run sanitization + write outputs
Outputs:
- `sanidata/sc_x/**/<table>.parquet`
- `sanidata/sc_x/**//<table>.csv` (optional, for compatibility)


In [6]:
for sc, tables in INPUTS.items():
    for table, paths in tables.items():
        if table not in SANITIZERS:
            continue

        for p in paths:
            # preserve the relative folder structure under BASE_DIR
            rel_parent = p.parent.relative_to(BASE_DIR)

            out_dir = OUT_DIR.joinpath(rel_parent) # OUT_BASE = BASE_DIR / "sanidata"'
            out_dir.mkdir(parents=True, exist_ok=True)

            df = load_csv_raw(p)
            print(f"Sanitizing: sc={sc} table={table} file={p} shape={df.shape}")
            s = SANITIZERS[table](df)

            # output paths:
            # sanidata/sc_x/**/<table>.parquet and sanidata/sc_x/**/<table>.csv
            out_pq  = out_dir / f"{table}.parquet"
            out_csv = out_dir / f"{table}.csv"

            s.to_parquet(out_pq, index=False)
            s.to_csv(out_csv)

            s.to_parquet(out_pq, index=False)
            s.to_csv(out_csv, index=False)

print("Done. Files written under:", OUT_DIR.resolve())

# Quick peek (first few outputs)
print("Example outputs:", sorted([str(p) for p in OUT_DIR.rglob("*.parquet")])[:10])

Sanitizing: sc=sc1 table=conn file=/Users/zhuoran/Projects/SSCMDataset/data/sc1/windows/azure_conn.csv shape=(5746, 44)
Sanitizing: sc=sc1 table=events file=/Users/zhuoran/Projects/SSCMDataset/data/sc1/windows/azure_events.csv shape=(1000, 21)
Sanitizing: sc=sc1 table=perf file=/Users/zhuoran/Projects/SSCMDataset/data/sc1/windows/azure_perf.csv shape=(15000, 18)
Sanitizing: sc=sc1 table=port file=/Users/zhuoran/Projects/SSCMDataset/data/sc1/windows/azure_port.csv shape=(1109, 24)
Sanitizing: sc=sc1 table=process file=/Users/zhuoran/Projects/SSCMDataset/data/sc1/windows/azure_process.csv shape=(345, 27)
Sanitizing: sc=sc1 table=security file=/Users/zhuoran/Projects/SSCMDataset/data/sc1/windows/azure_security.csv shape=(334, 234)
Sanitizing: sc=sc6 table=events file=/Users/zhuoran/Projects/SSCMDataset/data/sc6/victim/azure_events.csv shape=(6528, 21)
Sanitizing: sc=sc6 table=syslog file=/Users/zhuoran/Projects/SSCMDataset/data/sc6/attacker/syslog.csv shape=(3246, 16)
Sanitizing: sc=sc5 t

## 6) Quick spot-check (raw vs sanitized)
This section samples a few rows to confirm that:
- Non-IP identifiers are tokenized
- IPs remain unchanged


In [7]:
import numpy as np

def sample_compare(sc: str, table: str, path: Path, n: int=3, use_parquet: bool=True):
    ''' compare one raw file vs its sanitized output.

    output path mirrors the input folder structure:
      out_dir = OUT_BASE / (path.parent relative to BASE_DIR)
      then <table>.parquet / <table>.csv
    '''
    a = load_csv_raw(path)

    rel_parent = path.parent.relative_to(BASE_DIR)
    out_dir = OUT_DIR / rel_parent
    out_pq = out_dir / f"{table}.parquet"
    out_csv = out_dir / f"{table}.csv"

    if use_parquet and out_pq.exists():
        b = pd.read_parquet(out_pq)
    elif out_csv.exists():
        b = pd.read_csv(out_csv, dtype=str, keep_default_na=False)
    else:
        raise FileNotFoundError(f"No sanitized output found for {sc}/{table} at {out_dir}")

    if len(a) == 0:
        print("Empty raw table:", sc, table, path)
        return

    idx = np.random.choice(len(a), size=min(n, len(a)), replace=False)
    cols = [c for c in a.columns if c in b.columns]

    print(f"\n=== sc={sc} table={table} ===")
    print("raw:", path)
    print("san:", out_pq if (use_parquet and out_pq.exists()) else out_csv)

    display(pd.concat(
        [a.iloc[idx][cols].assign(_version="raw"),
         b.iloc[idx][cols].assign(_version="sanitized")],
        axis=0
    ).sort_values("_version"))


In [8]:
tables_to_check = ["conn", "events", "process", "syslog"]

for sc, tables in INPUTS.items():
    for table in tables_to_check:
        if table not in tables:
            continue
        # pick one file path (if multiple)
        p = tables[table][0]
        sample_compare(sc, table, p, n=2)
    break


=== sc=sc1 table=conn ===
raw: /Users/zhuoran/Projects/SSCMDataset/data/sc1/windows/azure_conn.csv
san: /Users/zhuoran/Projects/SSCMDataset/sanidata/sc1/windows/conn.parquet


,TimeGenerated [UTC],Computer,Direction,ProcessName,SourceIp,DestinationIp,DestinationPort,Protocol,RemoteIp,RemoteDnsQuestions,RemoteDnsCanonicalNames,RemoteClassification,RemoteLongitude,RemoteLatitude,RemoteCountry,BytesSent,BytesReceived,LinksLive,LinksTerminated,LinksEstablished,LinksFailed,Responses,ResponseTimeSum,ResponseTimeMin,ResponseTimeMax,MaliciousIp,IndicatorThreatType,Description,TLPLevel,Confidence,Severity,FirstReportedDateTime,LastReportedDateTime,IsActive,ReportReferenceLink,AdditionalInformation,ConnectionId,Machine,Process,AgentId,TenantId,SourceSystem,Type,_ResourceId,_version
4059,"22/04/2025, 15:07:00.143",Office-Wins,outbound,HealthService,10.0.0.5,51.140.148.51,443,tcp,51.140.148.51,"[""62f68ee4-9639-4a69-b820-5881cfe41a62.ods.opinsights.azure.com"",""8f3c4b92-3deb-4244-a26b-bf9f3a4ca94f.ods.opinsights.azure.com""]","[""ipv4-suk-oi-ods-cses-b.uksouth.cloudapp.azure.com""]",,-0.13,51.5,United Kingdom,34605,1904,4,0,0,0,7,488,0,125,,,,,,,,,,,,19B5F8B32671B4968FA8847101AFDC0572234C3A,m-16ca1849-0880-47c2-8569-c58eee5b8489,p-2030ac041acf2f26393f0a00bdce62b358ae1f15,16ca1849-0880-47c2-8569-c58eee5b8489,8f3c4b92-3deb-4244-a26b-bf9f3a4ca94f,OpsManager,VMConnection,/subscriptions/125ecc14-76ae-4cae-918e-d8da5f3a4d51/resourcegroups/rg-uks-prod-shotgun/providers/microsoft.compute/virtualmachines/offic...,raw
1562,"22/04/2025, 13:36:00.040",Office-Wins,outbound,SearchApp,10.0.0.5,2.18.27.82,443,tcp,2.18.27.82,"[""th.bing.com"",""www.bing.com""]","[""a2-18-27-82.deploy.static.akamaitechnologies.com"",""e86303.dscx.akamaiedge.net""]",,-0.6,51.51,United Kingdom,0,0,1,0,0,0,0,,,,,,,,,,,,,,,7CFB4FEA90243050DD8D0885E5A14F51DF98F9FD,m-16ca1849-0880-47c2-8569-c58eee5b8489,p-b95dbddc54fcaad049f80db5f522ae5b06945778,16ca1849-0880-47c2-8569-c58eee5b8489,8f3c4b92-3deb-4244-a26b-bf9f3a4ca94f,OpsManager,VMConnection,/subscriptions/125ecc14-76ae-4cae-918e-d8da5f3a4d51/resourcegroups/rg-uks-prod-shotgun/providers/microsoft.compute/virtualmachines/offic...,raw
4059,"22/04/2025, 15:07:00.143",HOST_760,outbound,HealthService,10.0.0.5,51.140.148.51,443,tcp,51.140.148.51,"[""GUID_9576.FQDN_8215.example"",""GUID_1544.FQDN_8215.example""]","[""FQDN_1175.example""]",,-0.13,51.5,United Kingdom,34605,1904,4,0,0,0,7,488,0,125,,,,,,,,,,,,19B5F8B32671B4968FA8847101AFDC0572234C3A,HOST_975,p-2030ac041acf2f26393f0a00bdce62b358ae1f15,AGENT_288,TEN_544,OpsManager,VMConnection,/subscriptions/SUB_785/resourceGroups/RG_081/providers/microsoft.compute/virtualMachines/VM_021,sanitized
1562,"22/04/2025, 13:36:00.040",HOST_760,outbound,SearchApp,10.0.0.5,2.18.27.82,443,tcp,2.18.27.82,"[""FQDN_1836.example"",""FQDN_2365.example""]","[""FQDN_2919.example"",""FQDN_8434.example""]",,-0.6,51.51,United Kingdom,0,0,1,0,0,0,0,,,,,,,,,,,,,,,7CFB4FEA90243050DD8D0885E5A14F51DF98F9FD,HOST_975,p-b95dbddc54fcaad049f80db5f522ae5b06945778,AGENT_288,TEN_544,OpsManager,VMConnection,/subscriptions/SUB_785/resourceGroups/RG_081/providers/microsoft.compute/virtualMachines/VM_021,sanitized



=== sc=sc1 table=events ===
raw: /Users/zhuoran/Projects/SSCMDataset/data/sc1/windows/azure_events.csv
san: /Users/zhuoran/Projects/SSCMDataset/sanidata/sc1/windows/events.parquet


,TenantId,SourceSystem,TimeGenerated [UTC],Source,EventLog,Computer,EventLevel,EventLevelName,ParameterXml,EventData,EventID,RenderedDescription,AzureDeploymentID,Role,EventCategory,UserName,Message,MG,ManagementGroupName,Type,_ResourceId,_version
979,62f68ee4-9639-4a69-b820-5881cfe41a62,OpsManager,"22/04/2025, 14:27:10.241",Microsoft-Windows-Sysmon,Microsoft-Windows-Sysmon/Operational,Office-Wins,4,Information,<Param>Usermode</Param><Param>2025-04-22 14:27:08.636</Param><Param>{d55f260b-982e-6807-e203-000000001d00}</Param><Param>1616</Param><Pa...,"<DataItem Type=""System.XmlData"" time=""2025-04-22T14:27:10.2414814Z"" sourceHealthServiceId=""028c967d-492d-4888-8522-cf90541bc093""><EventD...",3,Network connection detected: RuleName: Usermode UtcTime: 2025-04-22 14:27:08.636 ProcessGuid: {d55f260b-982e-6807-e203-000000001d00} Pro...,,,3,S-1-5-18,,00000000-0000-0000-0000-000000000001,AOI-62f68ee4-9639-4a69-b820-5881cfe41a62,Event,/subscriptions/125ecc14-76ae-4cae-918e-d8da5f3a4d51/resourcegroups/rg-uks-prod-shotgun/providers/microsoft.compute/virtualmachines/offic...,raw
579,62f68ee4-9639-4a69-b820-5881cfe41a62,OpsManager,"22/04/2025, 14:19:52.045",Microsoft-Windows-Sysmon,Microsoft-Windows-Sysmon/Operational,Office-Wins,4,Information,<Param>Usermode</Param><Param>2025-04-22 14:19:50.419</Param><Param>{d55f260b-982e-6807-e203-000000001d00}</Param><Param>1616</Param><Pa...,"<DataItem Type=""System.XmlData"" time=""2025-04-22T14:19:52.0455489Z"" sourceHealthServiceId=""028c967d-492d-4888-8522-cf90541bc093""><EventD...",3,Network connection detected: RuleName: Usermode UtcTime: 2025-04-22 14:19:50.419 ProcessGuid: {d55f260b-982e-6807-e203-000000001d00} Pro...,,,3,S-1-5-18,,00000000-0000-0000-0000-000000000001,AOI-62f68ee4-9639-4a69-b820-5881cfe41a62,Event,/subscriptions/125ecc14-76ae-4cae-918e-d8da5f3a4d51/resourcegroups/rg-uks-prod-shotgun/providers/microsoft.compute/virtualmachines/offic...,raw
979,TEN_576,OpsManager,"22/04/2025, 14:27:10.241",Microsoft-Windows-Sysmon,Microsoft-Windows-Sysmon/Operational,HOST_760,4,Information,<Param>Usermode</Param><Param>2025-04-22 14:27:08.636</Param><Param>{d55f260b-982e-6807-e203-000000001d00}</Param><Param>1616</Param><Pa...,"<DataItem Type=""System.XmlData"" time=""2025-04-22T14:27:10.2414814Z"" sourceHealthServiceId=""028c967d-492d-4888-8522-cf90541bc093""><EventD...",3,Network connection detected: RuleName: Usermode UtcTime: 2025-04-22 14:27:08.636 ProcessGuid: {GUID_2708} ProcessId: 1616 Image: C:\User...,,,3,USER_550,,MG_953,MGN_934,Event,/subscriptions/SUB_785/resourceGroups/RG_081/providers/microsoft.compute/virtualMachines/VM_021,sanitized
579,TEN_576,OpsManager,"22/04/2025, 14:19:52.045",Microsoft-Windows-Sysmon,Microsoft-Windows-Sysmon/Operational,HOST_760,4,Information,<Param>Usermode</Param><Param>2025-04-22 14:19:50.419</Param><Param>{d55f260b-982e-6807-e203-000000001d00}</Param><Param>1616</Param><Pa...,"<DataItem Type=""System.XmlData"" time=""2025-04-22T14:19:52.0455489Z"" sourceHealthServiceId=""028c967d-492d-4888-8522-cf90541bc093""><EventD...",3,Network connection detected: RuleName: Usermode UtcTime: 2025-04-22 14:19:50.419 ProcessGuid: {GUID_2708} ProcessId: 1616 Image: C:\User...,,,3,USER_550,,MG_953,MGN_934,Event,/subscriptions/SUB_785/resourceGroups/RG_081/providers/microsoft.compute/virtualMachines/VM_021,sanitized



=== sc=sc1 table=process ===
raw: /Users/zhuoran/Projects/SSCMDataset/data/sc1/windows/azure_process.csv
san: /Users/zhuoran/Projects/SSCMDataset/sanidata/sc1/windows/process.parquet


,TimeGenerated [UTC],Computer,AgentId,Machine,Process,ExecutableName,DisplayName,Role,Group,StartTime [UTC],FirstPid,Description,CompanyName,InternalName,ProductName,ProductVersion,FileVersion,ExecutablePath,CommandLine,WorkingDirectory,Services,UserName,UserDomain,TenantId,SourceSystem,Type,_ResourceId,_version
156,"22/04/2025, 14:01:00.058",Office-Wins,16ca1849-0880-47c2-8569-c58eee5b8489,m-16ca1849-0880-47c2-8569-c58eee5b8489,p-952b9c461553e264d4858f5ef88aeb4bd08b2e5b,python3,python3,,Python,"22/04/2025, 14:01:06.547",11060,Python,Python Software Foundation,Python Console,Python,3.10.11,3.10.11,C:\Users\Newt\.pyenv\pyenv-win\versions\3.10.11\python3.exe,"""C:\Users\Newt\.pyenv\pyenv-win\versions\3.10.11\python3.exe"" ""-c"" ""from multiprocessing.spawn import spawn_main; spawn_main(parent_pid=...",C:\package\sc1\,,Newt,Office-Wins,8f3c4b92-3deb-4244-a26b-bf9f3a4ca94f,Insights,VMProcess,/subscriptions/125ecc14-76ae-4cae-918e-d8da5f3a4d51/resourcegroups/rg-uks-prod-shotgun/providers/microsoft.compute/virtualmachines/offic...,raw
58,"22/04/2025, 12:58:00.028",Office-Wins,16ca1849-0880-47c2-8569-c58eee5b8489,m-16ca1849-0880-47c2-8569-c58eee5b8489,p-20733516290305ff7afcef5ac7d405d42a5807e0,backgroundTaskHost,backgroundTaskHost,,Microsoft® Windows® Operating System,"22/04/2025, 12:58:08.751",11704,Background Task Host,Microsoft Corporation,Background Task Host,Microsoft® Windows® Operating System,10.0.19041.3636,10.0.19041.3636 (WinBuild.160101.0800),C:\Windows\system32\backgroundTaskHost.exe,"""C:\Windows\system32\backgroundTaskHost.exe"" -ServerName:App.AppXmtcan0h2tfbfy7k9kn8hbxb6dmzz1zh0.mca",C:\Windows\SystemApps\Microsoft.Windows.ContentDeliveryManager_cw5n1h2txyewy\,,Newt,Office-Wins,8f3c4b92-3deb-4244-a26b-bf9f3a4ca94f,Insights,VMProcess,/subscriptions/125ecc14-76ae-4cae-918e-d8da5f3a4d51/resourcegroups/rg-uks-prod-shotgun/providers/microsoft.compute/virtualmachines/offic...,raw
156,"22/04/2025, 14:01:00.058",HOST_760,AGENT_288,HOST_975,p-952b9c461553e264d4858f5ef88aeb4bd08b2e5b,python3,python3,,Python,"22/04/2025, 14:01:06.547",11060,Python,Python Software Foundation,Python Console,Python,3.10.11,3.10.11,C:\Users\Newt\.pyenv\pyenv-win\versions\3.10.11\python3.exe,"""C:\Users\Newt\.pyenv\pyenv-win\versions\3.10.11\python3.exe"" ""-c"" ""from FQDN_0879.example import spawn_main; spawn_main(parent_pid=1114...",C:\package\sc1\,,USER_474,USER_760,TEN_544,Insights,VMProcess,/subscriptions/SUB_785/resourceGroups/RG_081/providers/microsoft.compute/virtualMachines/VM_021,sanitized
58,"22/04/2025, 12:58:00.028",HOST_760,AGENT_288,HOST_975,p-20733516290305ff7afcef5ac7d405d42a5807e0,backgroundTaskHost,backgroundTaskHost,,Microsoft® Windows® Operating System,"22/04/2025, 12:58:08.751",11704,Background Task Host,Microsoft Corporation,Background Task Host,Microsoft® Windows® Operating System,10.0.19041.3636,10.0.19041.3636 (WinBuild.160101.0800),C:\Windows\system32\backgroundTaskHost.exe,"""C:\Windows\system32\backgroundTaskHost.exe"" -ServerName:FQDN_3709.example",C:\Windows\SystemApps\Microsoft.Windows.ContentDeliveryManager_cw5n1h2txyewy\,,USER_474,USER_760,TEN_544,Insights,VMProcess,/subscriptions/SUB_785/resourceGroups/RG_081/providers/microsoft.compute/virtualMachines/VM_021,sanitized


## 7) Optional leak checks
Search for strings you expect to be removed (e.g., organization domains, old nicknames, raw subscription/resource IDs).
Add them to `needles` and run this scan over the sanitized tables.

Note: IPs are not processed, so avoid using IP needles unless you explicitly want to confirm that they remain present.

In [9]:
def find_leaks(df: pd.DataFrame, needles):
    hits = []
    for col in df.columns:
        if df[col].dtype == object:
            s = df[col].astype(str)
            for needle in needles:
                mask = s.str.contains(re.escape(needle), na=False)
                if mask.any():
                    hits.append((col, needle, int(mask.sum())))
    return hits


needles = [
    "gla.ac.uk",
    "subscriptions/",
    # "azureofficer",
]

def read_sanitized_file(path: Path) -> pd.DataFrame:
    if path.suffix == ".parquet":
        return pd.read_parquet(path)
    # keep as str for consistent matching
    return pd.read_csv(path, dtype=str, keep_default_na=False)

# Scan all sanitized outputs under OUT_BASE (e.g., data/sanidata/)
for f in sorted(OUT_DIR.rglob("*.parquet")) + sorted(OUT_DIR.rglob("*.csv")):
    df = read_sanitized_file(f)
    hits = find_leaks(df, needles)
    if hits:
        rel = f.relative_to(OUT_DIR)
        print(f"\nPotential occurrences in: {rel}")
        for col, needle, cnt in hits[:50]:
            print(f"  col={col} needle={needle} count={cnt}")



Potential occurrences in: sc1/windows/conn.parquet
  col=_ResourceId needle=subscriptions/ count=5746

Potential occurrences in: sc1/windows/events.parquet
  col=_ResourceId needle=subscriptions/ count=1000

Potential occurrences in: sc1/windows/perf.parquet
  col=_ResourceId needle=subscriptions/ count=15000

Potential occurrences in: sc1/windows/port.parquet
  col=_ResourceId needle=subscriptions/ count=1109

Potential occurrences in: sc1/windows/process.parquet
  col=_ResourceId needle=subscriptions/ count=345

Potential occurrences in: sc1/windows/security.parquet
  col=_ResourceId needle=subscriptions/ count=334

Potential occurrences in: sc2/attacker/syslog.parquet
  col=_ResourceId needle=subscriptions/ count=9732

Potential occurrences in: sc2/events.parquet
  col=ParameterXml needle=gla.ac.uk count=4
  col=EventData needle=gla.ac.uk count=4
  col=_ResourceId needle=subscriptions/ count=44246

Potential occurrences in: sc3/syslog.parquet
  col=SyslogMessage needle=subscription